In [1]:
from pathlib import Path
import sqlite3
import pandas as pd

# Carpeta donde se guardará la base del Día 5.
CARPETA_DATOS = Path("datos_dia_05")
CARPETA_DATOS.mkdir(exist_ok=True)

# Ruta de la base de datos.
RUTA_DB = CARPETA_DATOS / "ventas_dia_05.db"

print("Carpeta de trabajo:")
print(CARPETA_DATOS.resolve())

print("\nBase de datos:")
print(RUTA_DB.resolve())

Carpeta de trabajo:
C:\Users\alumn\OneDrive\Escritorio\Cursos 2026-02\Persistencia de datos con python\Códigos\notebooks\datos_dia_05

Base de datos:
C:\Users\alumn\OneDrive\Escritorio\Cursos 2026-02\Persistencia de datos con python\Códigos\notebooks\datos_dia_05\ventas_dia_05.db


In [2]:
conexion = sqlite3.connect(RUTA_DB)
cursor = conexion.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS productos (
    id_producto INTEGER PRIMARY KEY AUTOINCREMENT,
    sku TEXT NOT NULL UNIQUE,
    nombre TEXT NOT NULL,
    precio REAL NOT NULL CHECK(precio > 0),
    stock INTEGER NOT NULL DEFAULT 0 CHECK(stock >= 0),
    categoria TEXT NOT NULL DEFAULT 'General',
    activo INTEGER NOT NULL DEFAULT 1 CHECK(activo IN (0, 1))
)
""")

conexion.commit()
conexion.close()

print("Tabla productos creada correctamente.")

Tabla productos creada correctamente.


In [29]:
conexion = sqlite3.connect(RUTA_DB)
cursor = conexion.cursor()

productos = [
    ("P001", "Mouse inalámbrico", 249.90, 15, "Accesorios", 1),
    ("P002", "Teclado mecánico", 899.00, 8, "Accesorios", 1),
    ("P003", "Monitor 24 pulgadas", 3299.00, 4, "Pantallas", 1),
    ("P004", "Cable HDMI", 120.00, 20, "Cables", 1),
    ("P005", "Memoria USB 64GB", 150.00, 30, "Almacenamiento", 1),
    ("P006", "Laptop básica", 9500.00, 3, "Computo", 1),
    ("P007", "Adaptador USB-C", 199.00, 12, "Accesorios", 1),
    ("P008", "Disco duro externo 1TB", 1200.00, 5, "Almacenamiento", 1),
    ("P009", "Bocinas Bluetooth", 699.00, 0, "Audio", 1),
    ("P010", "Audífonos alámbricos", 180.00, 25, "Audio", 1)
]

for producto in productos:
    try:
        cursor.execute("""
        INSERT INTO productos (sku, nombre, precio, stock, categoria, activo)
        VALUES (?, ?, ?, ?, ?, ?)
        """, producto)
    except sqlite3.IntegrityError:
        # Si el SKU ya existe, no detenemos la ejecución del notebook.
        pass

conexion.commit()
conexion.close()

print("Productos de ejemplo insertados correctamente.")

Productos de ejemplo insertados correctamente.


In [4]:
conexion = sqlite3.connect(RUTA_DB)

tabla_productos = pd.read_sql_query("""
SELECT id_producto, sku, nombre, precio, stock, categoria, activo
FROM productos
ORDER BY id_producto
""", conexion)

conexion.close()

tabla_productos

,id_producto,sku,nombre,precio,stock,categoria,activo
0,1,P001,Mouse inalámbrico,249.9,15,Accesorios,1
1,2,P002,Teclado mecánico,899.0,8,Accesorios,1
2,3,P003,Monitor 24 pulgadas,3299.0,4,Pantallas,1
3,4,P004,Cable HDMI,120.0,20,Cables,1
4,5,P005,Memoria USB 64GB,150.0,30,Almacenamiento,1
5,6,P006,Laptop básica,9500.0,3,Computo,1
6,7,P007,Adaptador USB-C,199.0,12,Accesorios,1
7,8,P008,Disco duro externo 1TB,1200.0,5,Almacenamiento,1
8,9,P009,Bocinas Bluetooth,699.0,0,Audio,1
9,10,P010,Audífonos alámbricos,180.0,25,Audio,1


In [ ]:
conexion = sqlite3.connect(RUTA_DB)

productos_stock_bajo = pd.read_sql_query("""
SELECT id_producto, sku, nombre, stock
FROM productos
WHERE stock < 10
ORDER BY stock ASC
""", conexion)
# =, =!, <, >, <=, >= 
conexion.close()

productos_stock_bajo

,id_producto,sku,nombre,stock
0,9,P009,Bocinas Bluetooth,0
1,6,P006,Laptop básica,3
2,3,P003,Monitor 24 pulgadas,4
3,8,P008,Disco duro externo 1TB,5
4,2,P002,Teclado mecánico,8


In [ ]:
conexion = sqlite3.connect(RUTA_DB)

productos_accesorios = pd.read_sql_query("""
SELECT id_producto, sku, nombre, precio, categoria
FROM productos
WHERE categoria = 'Accesorios' 
ORDER BY precio DESC
""", conexion)

conexion.close()

productos_accesorios

,id_producto,sku,nombre,precio,categoria
0,2,P002,Teclado mecánico,899.0,Accesorios
1,1,P001,Mouse inalámbrico,249.9,Accesorios
2,7,P007,Adaptador USB-C,199.0,Accesorios


In [ ]:
conexion = sqlite3.connect(RUTA_DB)

consulta_relacional = pd.read_sql_query("""
SELECT sku, nombre, precio, stock
FROM productos
WHERE precio >= 500
ORDER BY precio ASC
""", conexion)

conexion.close()

consulta_relacional

,sku,nombre,precio,stock
0,P009,Bocinas Bluetooth,699.0,0
1,P002,Teclado mecánico,899.0,8
2,P008,Disco duro externo 1TB,1200.0,5
3,P003,Monitor 24 pulgadas,3299.0,4
4,P006,Laptop básica,9500.0,3


In [8]:
conexion = sqlite3.connect(RUTA_DB)

productos_no_accesorios = pd.read_sql_query("""
SELECT sku, nombre, precio, categoria
FROM productos
WHERE categoria != 'Accesorios'
ORDER BY categoria ASC
""", conexion)

conexion.close()

productos_no_accesorios

,sku,nombre,precio,categoria
0,P005,Memoria USB 64GB,150.0,Almacenamiento
1,P008,Disco duro externo 1TB,1200.0,Almacenamiento
2,P009,Bocinas Bluetooth,699.0,Audio
3,P010,Audífonos alámbricos,180.0,Audio
4,P004,Cable HDMI,120.0,Cables
5,P006,Laptop básica,9500.0,Computo
6,P003,Monitor 24 pulgadas,3299.0,Pantallas


In [ ]:
# AND, OR, NOT 
conexion = sqlite3.connect(RUTA_DB)

consulta_and = pd.read_sql_query("""
SELECT sku, nombre, precio, stock, categoria
FROM productos
WHERE stock < 10 AND precio > 500 AND categoria != 'Accesorios' 
ORDER BY precio DESC
""", conexion)

conexion.close()

consulta_and

,sku,nombre,precio,stock,categoria
0,P006,Laptop básica,9500.0,3,Computo
1,P003,Monitor 24 pulgadas,3299.0,4,Pantallas
2,P008,Disco duro externo 1TB,1200.0,5,Almacenamiento
3,P009,Bocinas Bluetooth,699.0,0,Audio


In [32]:
conexion = sqlite3.connect(RUTA_DB)

consulta_or = pd.read_sql_query("""
SELECT sku, nombre, precio, stock, categoria
FROM productos
WHERE categoria = 'Audio' OR categoria = 'Almacenamiento' AND stock < 10
ORDER BY categoria, nombre
""", conexion)

conexion.close()

consulta_or

,sku,nombre,precio,stock,categoria
0,P008,Disco duro externo 1TB,1200.0,5,Almacenamiento
1,P010,Audífonos alámbricos,180.0,25,Audio
2,P009,Bocinas Bluetooth,699.0,0,Audio


In [34]:
conexion = sqlite3.connect(RUTA_DB)

consulta_not = pd.read_sql_query("""
SELECT sku, nombre, precio, categoria
FROM productos
WHERE NOT categoria = 'Accesorios'
ORDER BY categoria ASC
""", conexion)

conexion.close()

consulta_not

,sku,nombre,precio,categoria
0,P005,Memoria USB 64GB,150.0,Almacenamiento
1,P008,Disco duro externo 1TB,1200.0,Almacenamiento
2,P010,Audífonos alámbricos,180.0,Audio
3,P009,Bocinas Bluetooth,699.0,Audio
4,P004,Cable HDMI,120.0,Cables
5,P006,Laptop básica,9500.0,Computo
6,P003,Monitor 24 pulgadas,3299.0,Pantallas


In [12]:
conexion = sqlite3.connect(RUTA_DB)

orden_precio_desc = pd.read_sql_query("""
SELECT sku, nombre, precio, stock
FROM productos
ORDER BY precio DESC
""", conexion)

conexion.close()

orden_precio_desc

,sku,nombre,precio,stock
0,P006,Laptop básica,9500.0,3
1,P003,Monitor 24 pulgadas,3299.0,4
2,P008,Disco duro externo 1TB,1200.0,5
3,P002,Teclado mecánico,899.0,8
4,P009,Bocinas Bluetooth,699.0,0
5,P001,Mouse inalámbrico,249.9,15
6,P007,Adaptador USB-C,199.0,12
7,P010,Audífonos alámbricos,180.0,25
8,P005,Memoria USB 64GB,150.0,30
9,P004,Cable HDMI,120.0,20


In [35]:
conexion = sqlite3.connect(RUTA_DB)

orden_multiple = pd.read_sql_query("""
SELECT sku, nombre, precio, stock, categoria
FROM productos
ORDER BY categoria ASC, precio DESC
""", conexion)

conexion.close()

orden_multiple

,sku,nombre,precio,stock,categoria
0,P002,Teclado mecánico,899.0,6,Accesorios
1,P001,Mouse inalámbrico,279.9,15,Accesorios
2,P007,Adaptador USB-C,199.0,12,Accesorios
3,P008,Disco duro externo 1TB,1200.0,5,Almacenamiento
4,P005,Memoria USB 64GB,150.0,30,Almacenamiento
5,P009,Bocinas Bluetooth,699.0,0,Audio
6,P010,Audífonos alámbricos,180.0,25,Audio
7,P004,Cable HDMI,120.0,20,Cables
8,P006,Laptop básica,9500.0,3,Computo
9,P003,Monitor 24 pulgadas,3299.0,4,Pantallas


In [14]:
conexion = sqlite3.connect(RUTA_DB)

producto_a_actualizar = pd.read_sql_query("""
SELECT id_producto, sku, nombre, precio, stock
FROM productos
WHERE sku = 'P001'
""", conexion)

conexion.close()

producto_a_actualizar

,id_producto,sku,nombre,precio,stock
0,1,P001,Mouse inalámbrico,249.9,15


In [41]:
conexion = sqlite3.connect(RUTA_DB)
cursor = conexion.cursor()

cursor.execute("""
UPDATE productos
SET precio = ?
WHERE sku = ?
""", (279.90, "P001"))

conexion.commit()
conexion.close()

print("Precio actualizado correctamente.")

Precio actualizado correctamente.


In [42]:
conexion = sqlite3.connect(RUTA_DB)

producto_actualizado = pd.read_sql_query("""
SELECT id_producto, sku, nombre, precio, stock
FROM productos
WHERE sku = 'P001'
""", conexion)

conexion.close()

producto_actualizado

,id_producto,sku,nombre,precio,stock
0,1,P001,Mouse inalámbrico,279.9,15


In [ ]:
conexion = sqlite3.connect(RUTA_DB)
cursor = conexion.cursor()

# Simulamos que se vendieron 2 unidades del producto P002.
cantidad_vendida = 2
sku_producto = "P002"

cursor.execute("""
UPDATE productos
SET stock = stock - ?
WHERE sku = ? AND stock >= ?
""", (cantidad_vendida, sku_producto, cantidad_vendida))

conexion.commit()

filas_modificadas = cursor.rowcount

conexion.close()

if filas_modificadas > 0:
    print("Stock actualizado correctamente.")
else:
    print("No se actualizó el stock. Puede que no exista el producto o no haya stock suficiente.")


No se actualizó el stock. Puede que no exista el producto o no haya stock suficiente.


In [ ]:
conexion = sqlite3.connect(RUTA_DB)
cursor = conexion.cursor()

cursor.execute("DROP TABLE IF EXISTS productos_prueba_update")

cursor.execute("""
CREATE TABLE productos_prueba_update (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre TEXT NOT NULL,
    stock INTEGER NOT NULL
)
""")

cursor.executemany("""
INSERT INTO productos_prueba_update (nombre, stock)
VALUES (?, ?)
""", [
    ("Producto A", 10),
    ("Producto B", 20),
    ("Producto C", 30)
])

conexion.commit()

antes = pd.read_sql_query("""
SELECT *
FROM productos_prueba_update
""", conexion)

print("Antes del UPDATE sin WHERE:")
display(antes)

cursor.execute("""
UPDATE productos_prueba_update
SET stock = 0
""")

conexion.commit()

despues = pd.read_sql_query("""
SELECT *
FROM productos_prueba_update
""", conexion)

conexion.close()

print("Después del UPDATE sin WHERE:")
display(despues)

Antes del UPDATE sin WHERE:


,id,nombre,stock
0,1,Producto A,10
1,2,Producto B,20
2,3,Producto C,30


Después del UPDATE sin WHERE:


,id,nombre,stock
0,1,Producto A,0
1,2,Producto B,0
2,3,Producto C,0


In [19]:
conexion = sqlite3.connect(RUTA_DB)

producto_a_eliminar = pd.read_sql_query("""
SELECT id_producto, sku, nombre, precio, stock, categoria
FROM productos
WHERE sku = 'P009'
""", conexion)

conexion.close()

producto_a_eliminar

,id_producto,sku,nombre,precio,stock,categoria
0,9,P009,Bocinas Bluetooth,699.0,0,Audio


In [ ]:
conexion = sqlite3.connect(RUTA_DB)
cursor = conexion.cursor()

cursor.execute("""
DELETE FROM productos
WHERE sku = ?
""", ("P009",))

conexion.commit()

filas_eliminadas = cursor.rowcount

conexion.close()

if filas_eliminadas > 0:
    print("Producto eliminado correctamente.")
else:
    print("No se eliminó ningún producto.")

Producto eliminado correctamente.


In [21]:
conexion = sqlite3.connect(RUTA_DB)

confirmar_eliminacion = pd.read_sql_query("""
SELECT id_producto, sku, nombre
FROM productos
WHERE sku = 'P009'
""", conexion)

conexion.close()

confirmar_eliminacion

,id_producto,sku,nombre


In [22]:
conexion = sqlite3.connect(RUTA_DB)
cursor = conexion.cursor()

cursor.execute("DROP TABLE IF EXISTS productos_prueba_delete")

cursor.execute("""
CREATE TABLE productos_prueba_delete (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre TEXT NOT NULL
)
""")

cursor.executemany("""
INSERT INTO productos_prueba_delete (nombre)
VALUES (?)
""", [
    ("Producto A",),
    ("Producto B",),
    ("Producto C",)
])

conexion.commit()

antes = pd.read_sql_query("""
SELECT *
FROM productos_prueba_delete
""", conexion)

print("Antes del DELETE sin WHERE:")
display(antes)

cursor.execute("""
DELETE FROM productos_prueba_delete
""")

conexion.commit()

despues = pd.read_sql_query("""
SELECT *
FROM productos_prueba_delete
""", conexion)

conexion.close()

print("Después del DELETE sin WHERE:")
display(despues)

Antes del DELETE sin WHERE:


,id,nombre
0,1,Producto A
1,2,Producto B
2,3,Producto C


Después del DELETE sin WHERE:


,id,nombre


In [25]:
conexion = sqlite3.connect(RUTA_DB)
cursor = conexion.cursor()

cursor.execute("""
UPDATE productos
SET activo = 0
WHERE sku = ?
""", ("P010",))

conexion.commit()
conexion.close()

print("Producto marcado como inactivo.")

Producto marcado como inactivo.


In [48]:
conexion = sqlite3.connect(RUTA_DB)

productos_activos = pd.read_sql_query("""
SELECT id_producto, sku, nombre, precio, stock, categoria, activo
FROM productos
WHERE activo = 1
ORDER BY id_producto
""", conexion)

conexion.close()

productos_activos

,id_producto,sku,nombre,precio,stock,categoria,activo
0,1,P001,Mouse inalámbrico,279.9,15,Accesorios,1
1,2,P002,Teclado mecánico,899.0,0,Accesorios,1
2,3,P003,Monitor 24 pulgadas,3299.0,4,Pantallas,1
3,4,P004,Cable HDMI,120.0,20,Cables,1
4,5,P005,Memoria USB 64GB,150.0,30,Almacenamiento,1
5,6,P006,Laptop básica,9500.0,3,Computo,1
6,7,P007,Adaptador USB-C,199.0,12,Accesorios,1
7,8,P008,Disco duro externo 1TB,1200.0,5,Almacenamiento,1
8,11,P009,Bocinas Bluetooth,699.0,0,Audio,1


In [49]:
conexion = sqlite3.connect(RUTA_DB)

productos_inactivos = pd.read_sql_query("""
SELECT *
FROM productos
WHERE activo = 0
""", conexion)

conexion.close()

productos_inactivos

,id_producto,sku,nombre,precio,stock,categoria,activo
0,10,P010,Audífonos alámbricos,180.0,25,Audio,0


In [50]:
from shutil import copy2
from datetime import datetime

marca_tiempo = datetime.now().strftime("%Y%m%d_%H%M%S")
RUTA_RESPALDO = CARPETA_DATOS / f"respaldo_ventas_dia_05_{marca_tiempo}.db"

copy2(RUTA_DB, RUTA_RESPALDO)

print("Respaldo creado correctamente:")
print(RUTA_RESPALDO.resolve())

Respaldo creado correctamente:
C:\Users\alumn\OneDrive\Escritorio\Cursos 2026-02\Persistencia de datos con python\Códigos\notebooks\datos_dia_05\respaldo_ventas_dia_05_20260612_154726.db
